# Proyecto Final: Aplicación de Visión por Computador
Este notebook es una plantilla editable para desarrollar un proyecto de visión por computador. La idea es que puedas adaptarlo tanto a clasificación de imágenes como a otras variantes relacionadas.

## Cómo usar esta plantilla
Antes de ejecutar el notebook completo, revisa estas decisiones:

1. Define qué problema vas a resolver.
2. Decide qué dataset vas a usar y cómo está organizado.
3. Ajusta las rutas, el número de clases y el tamaño de imagen.
4. Elige si vas a entrenar una CNN sencilla o usar `transfer learning`.

No todo el contenido es automáticamente ejecutable sin adaptar: algunas celdas son una guía para personalizar tu proyecto.

## 1. Descripción del proyecto
- ¿Qué problema quieres resolver?
- ¿Qué entrada usa el sistema: imágenes, vídeo, webcam?
- ¿Qué salida produce: clase, detección, segmentación, pose?
- ¿Por qué ese problema es interesante o útil?

In [ ]:
# Escribe aquí la descripción de tu proyecto
descripcion = """
Mi proyecto consiste en...
Problema que resuelvo:
Dataset elegido:
Objetivo principal:
"""

print(descripcion)

## 2. Configuración inicial del proyecto
Aquí debes ajustar las variables básicas antes de seguir.

In [ ]:
from pathlib import Path

# Configuración principal
PROJECT_NAME = "mi_proyecto_vision"
TASK_TYPE = "classification"  # classification, detection, segmentation, pose
DATASET_ROOT = Path("dataset")
TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR = DATASET_ROOT / "validation"
TEST_DIR = DATASET_ROOT / "test"

IMG_SIZE = (160, 160)
BATCH_SIZE = 32
NUM_CLASSES = 2  # cámbialo según tu problema
USE_TRANSFER_LEARNING = True

print("Proyecto:", PROJECT_NAME)
print("Tipo de tarea:", TASK_TYPE)
print("Dataset raíz:", DATASET_ROOT.resolve())

## 3. Preparación del conjunto de datos
Esta plantilla asume, por defecto, una estructura en carpetas por clase:

- `dataset/train/clase_1`
- `dataset/train/clase_2`
- `dataset/validation/clase_1`
- `dataset/validation/clase_2`

Si tu dataset no sigue ese formato, tendrás que adaptar la carga.

In [ ]:
# Subida en Colab o uso de archivos locales
try:
    from google.colab import files
    uploaded = files.upload()
except Exception:
    uploaded = {}
    print("No estás en Colab. Si trabajas en local, organiza el dataset en las rutas indicadas arriba.")

print("Train existe:", TRAIN_DIR.exists())
print("Validation existe:", VAL_DIR.exists())
print("Test existe:", TEST_DIR.exists())

In [ ]:
# Vista rápida de un ejemplo del dataset
import cv2
import matplotlib.pyplot as plt

example_candidates = list(TRAIN_DIR.rglob("*.jpg")) + list(TRAIN_DIR.rglob("*.png"))

if not example_candidates:
    print("No se ha encontrado ninguna imagen en TRAIN_DIR. Ajusta la ruta o prepara el dataset.")
else:
    example_path = example_candidates[0]
    img = cv2.imread(str(example_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(5, 5))
    plt.imshow(img_rgb)
    plt.title(f"Ejemplo: {example_path.parent.name}")
    plt.axis("off")
    plt.show()

## 4. Carga de datos para clasificación
Si estás haciendo clasificación y tu dataset está organizado por carpetas, esta celda te servirá como base.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

if TASK_TYPE != "classification":
    print("Esta celda está pensada para clasificación. Adáptala si tu proyecto es de otro tipo.")
else:
    if not TRAIN_DIR.exists() or not VAL_DIR.exists():
        raise FileNotFoundError("No se han encontrado TRAIN_DIR y VAL_DIR. Revisa la configuración.")

    train_gen = ImageDataGenerator(
        preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True
    )

    val_gen = ImageDataGenerator(
        preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
    )

    train_data = train_gen.flow_from_directory(
        TRAIN_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    val_data = val_gen.flow_from_directory(
        VAL_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    print("Clases detectadas:", train_data.class_indices)

## 5. Modelo base
Puedes partir de una CNN sencilla o de un modelo preentrenado. Aquí se deja preparado el caso de `transfer learning`, que suele ser una buena primera opción.

In [ ]:
from tensorflow.keras import layers, models

if USE_TRANSFER_LEARNING:
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])
else:
    model = models.Sequential([
        layers.Input(shape=IMG_SIZE + (3,)),
        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 6. Entrenamiento

In [ ]:
# Ejecuta esta celda cuando train_data y val_data estén listos
if TASK_TYPE == "classification" and "train_data" in globals() and "val_data" in globals():
    history = model.fit(train_data, validation_data=val_data, epochs=5)
else:
    print("Prepara primero los datos o adapta esta sección a tu tipo de tarea.")

## 7. Evaluación y visualización
Como mínimo, conviene incluir:
- curvas de entrenamiento
- precisión y pérdida
- matriz de confusión si procede
- ejemplos de aciertos y errores

In [ ]:
if "history" in globals():
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history["accuracy"], label="train")
    plt.plot(history.history["val_accuracy"], label="val")
    plt.title("Precisión")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history["loss"], label="train")
    plt.plot(history.history["val_loss"], label="val")
    plt.title("Pérdida")
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("Todavía no hay historial de entrenamiento.")

## 8. Conclusiones y mejoras futuras
En esta parte no basta con decir si el modelo ha ido bien o mal. Conviene explicar:

- qué decisiones tomaste sobre los datos
- qué arquitectura elegiste y por qué
- qué limitaciones tiene tu solución
- qué mejorarías si tuvieras más tiempo o más datos

In [ ]:
conclusiones = """
Lo que mejor ha funcionado ha sido...
La principal limitación del proyecto es...
Si continuara el trabajo, mejoraría...
"""

print(conclusiones)